In [ ]:
import requests, zipfile
from pathlib import Path

url = "https://www.nass.usda.gov/Research_and_Science/Cropland/Release/datasets/2018_30m_cdls.zip"

out_zip = Path("2018_30m_cdls.zip")

print("Downloading 2018 CDL (~1.8 GB)...")
with requests.get(url, stream=True, timeout=600) as r:
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    downloaded = 0
    with open(out_zip, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            print(f"\r  {downloaded/1e6:.0f} MB / {total/1e6:.0f} MB", end="")

print(f"\nSaved → {out_zip}")

# ── Extract ───────────────────────────────────────────────────────────────────
print("Extracting...")
with zipfile.ZipFile(out_zip, "r") as z:
    z.extractall("cdl_2018/")
    print("Files:", z.namelist())

# ── Locate the raster ─────────────────────────────────────────────────────────
cdl_files = list(Path("cdl_2018/").glob("*.img")) + \
            list(Path("cdl_2018/").glob("*.tif"))
print("CDL raster:", cdl_files[0])

  1874 MB / 1874 MB
Saved → 2018_30m_cdls.zip
Extracting...
Files: ['2018_30m_cdls.aux', '2018_30m_cdls.tfw', '2018_30m_cdls.tif', '2018_30m_cdls.tif.ovr', 'Metadata_Cropland-Data-Layer.htm']
CDL raster: cdl_2018/2018_30m_cdls.tif


In [2]:
# Step 2 — Get the Bounding Box of All USA Chips
import rasterio
import numpy as np
from pathlib import Path
from rasterio.warp import transform_bounds

label_dir = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/LabelHand")

usa_chips = sorted(label_dir.glob("USA_*_LabelHand.tif"))

# Get the combined bounding box of all USA chips
all_bounds = []
for chip_path in usa_chips:
    with rasterio.open(chip_path) as src:
        bounds = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
        all_bounds.append(bounds)

# Union of all chip bounds — use this to download CDL once
min_lon = min(b[0] for b in all_bounds)
min_lat = min(b[1] for b in all_bounds)
max_lon = max(b[2] for b in all_bounds)
max_lat = max(b[3] for b in all_bounds)

print(f"USA chips bbox: [{min_lon:.4f}, {min_lat:.4f}, {max_lon:.4f}, {max_lat:.4f}]")

USA chips bbox: [-95.6670, 38.4048, -94.4252, 40.2905]


In [ ]:
import rioxarray
from pyproj import Transformer
from pathlib import Path

def clip_cdl_to_aoi(cdl_path, min_lon, min_lat, max_lon, max_lat, out_path="cdl_usa_flood.tif"):

    t = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)
    min_x, min_y = t.transform(min_lon, min_lat)
    max_x, max_y = t.transform(max_lon, max_lat)
    buf = 10000

    print(f"Opening {cdl_path}...")
    # ── Remove chunks= argument entirely ─────────────────────────────────────
    da = rioxarray.open_rasterio(cdl_path, masked=True)

    print("Clipping to AOI...")
    clipped = da.rio.clip_box(
        minx=min_x - buf, miny=min_y - buf,
        maxx=max_x + buf, maxy=max_y + buf
    )

    # clipped.rio.to_raster(out_path, compress="lzw")
    print(f"Saved → {out_path}  shape={clipped.shape}")
    return out_path

clip_cdl_to_aoi(
    cdl_path = list(Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/cdl_2018/").glob("*.tif"))[0] ,
    min_lon = -95.6670,  
    min_lat =  38.4048,
    max_lon = -94.4252,
    max_lat =  40.2905,
    out_path= "/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/cdl_usa_flood.tif"
)

Opening cdl_2018/2018_30m_cdls.tif...
Clipping to AOI...
Saved → cdl_usa_flood.tif  shape=(1, 7747, 4131)


'cdl_usa_flood.tif'

In [ ]:
# Step 3 — Filter out USA Chips that has x% distribution of Flood in CropLand
import rasterio
import numpy as np
from rasterio.warp import transform_bounds, reproject, Resampling
from pathlib import Path

CROPLAND_CODES = set(range(1, 62)) | set(range(66, 78)) | set(range(204, 255))
CDL_PATH    = "/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/generated/cdl_usa_flood.tif"
LABEL_DIR =  Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/LabelHand")
THRESHOLD   = 0.15   # see note below on threshold

def get_flooded_cropland_fraction(label_path, cdl_path, cropland_codes):
    """
    Returns fraction of FLOODED pixels (label==1) that overlap cropland in CDL.
    This is the scientifically correct filter for agricultural flood damage.
    """
    # Load label chip
    with rasterio.open(label_path) as src:
        label     = src.read(1)          # (512, 512)
        chip_crs  = src.crs
        chip_transform = src.transform
        chip_shape = label.shape

    # Load CDL clipped to chip bounds
    with rasterio.open(label_path) as src:
        bounds_4326 = transform_bounds(src.crs, "EPSG:4326", *src.bounds)

    with rasterio.open(cdl_path) as cdl_src:
        bounds_cdl = transform_bounds(
            "EPSG:4326", cdl_src.crs, *bounds_4326
        )
        window = cdl_src.window(*bounds_cdl)
        cdl_raw = cdl_src.read(1, window=window)
        cdl_transform = cdl_src.window_transform(window)
        cdl_crs = cdl_src.crs

    # Reproject CDL to match label chip exactly
    cdl_reprojected = np.zeros(chip_shape, dtype=np.uint8)
    reproject(
        source      = cdl_raw,
        destination = cdl_reprojected,
        src_transform  = cdl_transform,
        src_crs        = cdl_crs,
        dst_transform  = chip_transform,
        dst_crs        = chip_crs,
        resampling     = Resampling.nearest
    )

    # Key masks
    flood_mask    = (label == 1)
    cropland_mask = np.isin(cdl_reprojected, list(cropland_codes))

    flood_px = np.sum(flood_mask)
    if flood_px == 0:
        return 0.0, 0, 0   # no flood pixels at all

    # Flooded cropland = pixels that are BOTH flooded AND cropland
    flooded_crop_px = np.sum(flood_mask & cropland_mask)
    fraction = float(flooded_crop_px) / float(flood_px)

    return fraction, int(flooded_crop_px), int(flood_px)


# ── Run across all USA chips ──────────────────────────────────────────────────
from pathlib import Path
import pandas as pd

label_dir = Path(LABEL_DIR)
results = []

for label_path in sorted(label_dir.glob("USA_*_LabelHand.tif")):
    frac, flooded_crop_px, total_flood_px = get_flooded_cropland_fraction(
        label_path, CDL_PATH, CROPLAND_CODES
    )
    chip_id = label_path.name.replace("_LabelHand.tif", "")
    keep = frac >= THRESHOLD
    results.append({
        "chip_id":             chip_id,
        "flooded_crop_frac":   round(frac, 3),
        "flooded_crop_px":     flooded_crop_px,
        "total_flood_px":      total_flood_px,
        "keep":                keep
    })
    # print(f"{chip_id:30s}  flooded_crop={frac:.1%}  "
    #       f"({flooded_crop_px}/{total_flood_px} flood px)  "
    #       f"{'✓' if keep else '✗'}")

df = pd.DataFrame(results).sort_values("flooded_crop_frac", ascending=False)
kept = df[df["keep"]]
print(f"\nKept {len(kept)}/{len(df)} chips with ≥{THRESHOLD:.0%} "
      f"flooded pixels on cropland")
FILTER_CHIP_FILENAME = "usa_flooded_cropland_chips_15%.csv"
# kept[["chip_id","flooded_crop_frac","flooded_crop_px"]].to_csv(, index=False)
print(kept[["chip_id","flooded_crop_frac","flooded_crop_px"]].to_string(index=False))
# chip_ids = kept["chip_id"].tolist()
# print(f"Processing {len(chip_ids)} chips saved in {FILTER_CHIP_FILENAME} CSV")


Kept 23/69 chips with ≥15% flooded pixels on cropland
    chip_id  flooded_crop_frac  flooded_crop_px
 USA_430764              0.861            13289
 USA_955053              0.767            27479
  USA_58086              0.763            12130
  USA_86502              0.748            49655
 USA_170264              0.742            32290
 USA_595451              0.732            15028
 USA_217598              0.691            31254
 USA_826217              0.680             3060
 USA_646878              0.665             1613
USA_1068362              0.659            20541
 USA_986268              0.654             5949
 USA_638521              0.329             1701
 USA_224165              0.317               51
 USA_831672              0.316             6629
 USA_652955              0.312             2664
 USA_770353              0.287             9881
  USA_11422              0.275             9919
 USA_761032              0.227             2753
 USA_741178              0.204   

In [10]:
#4 Get bounding box and other metadata for all the filtered chips 
import rasterio
import geopandas as gpd
import pandas as pd
import json
from pathlib import Path
from shapely.geometry import box
from rasterio.warp import transform_bounds

# ── Inputs ────────────────────────────────────────────────────────────────────
FILTERED_CSV  = Path(FILTER_CHIP_FILENAME)
LABEL_DIR =  Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/data/flood_events/HandLabeled/LabelHand")
METADATA_PATH = Path("/users/PGS0218/julina/projects/geography/damage_mapping_terramind/Flood_to_cropland_ETL/Sen1Floods11_data/v1.1/Sen1Floods11_Metadata.geojson")
# ── Load metadata and build date lookup ───────────────────────────────────────
with open(METADATA_PATH) as f:
    meta = json.load(f)
    
# Inspect raw keys first so we pick the right field names
print("\nMetadata fields available:")
print(json.dumps(meta["features"][0]["properties"], indent=2))

event_dates = {}
for feature in meta["features"]:
    props   = feature["properties"]
    country = props.get("location") or props.get("Country")
    rel_orbit_num = props.get("rel_orbit_num")
    orbit = props.get("orbit")
    if country:
        event_dates[country] = {
            "s1_date": props.get('s1_date', "unknown"),
            "s2_date": props.get('s2_date', "unknown"),
            "rel_orbit_num": rel_orbit_num,
            "orbit": orbit
        }
print("Events found in metadata:")
for k, v in event_dates.items():
    print(f"  {k:20s} → {v}")


Metadata fields available:
{
  "coincident_size": 10.0,
  "location": "Bolivia",
  "orbit": "DESCENDING",
  "rel_orbit_num": 156.0,
  "s1_date": "2018/02/15",
  "s2_date": "2018/02/15",
  "ID": 1,
  "ISO_CC": "BOL",
  "VH_thresh": -20.44,
  "train_chip": 224,
  "val_chip": 15
}
Events found in metadata:
  Bolivia              → {'s1_date': '2018/02/15', 's2_date': '2018/02/15', 'rel_orbit_num': 156.0, 'orbit': 'DESCENDING'}
  Colombia             → {'s1_date': '2018/08/22', 's2_date': '2018/08/23', 'rel_orbit_num': 106.0, 'orbit': 'ASCENDING'}
  Ghana                → {'s1_date': '2018/09/18', 's2_date': '2018/09/19', 'rel_orbit_num': 147.0, 'orbit': 'ASCENDING'}
  India                → {'s1_date': '2016/08/12', 's2_date': '2016/08/12', 'rel_orbit_num': 77.0, 'orbit': 'DESCENDING'}
  Cambodia             → {'s1_date': '2018/08/05', 's2_date': '2018/08/04', 'rel_orbit_num': 26.0, 'orbit': 'ASCENDING'}
  Nigeria              → {'s1_date': '2018/09/21', 's2_date': '2018/09/20', 'rel_orb

In [12]:
# ── Extract chip geometry from GeoTIFF bounds ─────────────────────────────────
def get_chip_geometry(label_path):
    """Returns chip bounding box as a Shapely polygon in EPSG:4326."""
    with rasterio.open(label_path) as src:
        from rasterio.warp import transform_bounds
        bounds = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
    return box(*bounds)   # (minx, miny, maxx, maxy)


# ── Load your filtered chip list ──────────────────────────────────────────────
# filtered = pd.read_csv(FILTERED_CSV)
filtered = kept[["chip_id","flooded_crop_frac","flooded_crop_px"]]
chip_ids = filtered["chip_id"].tolist()
print(f"Processing {len(chip_ids)} chips from filtered {FILTERED_CSV} CSV")

records = []
for chip_id in chip_ids:
    label_path = LABEL_DIR / f"{chip_id}_LabelHand.tif"
    # Geometry
    geom = get_chip_geometry(label_path)
    b = geom.bounds

    # Dates from metadata
    country = chip_id.split("_")[0]
    event   = event_dates.get(country, {"s1_date": "unknown", "s2_date": "unknown", "rel_orbit_num": "unknown", "orbit": "unknown"})

    records.append({
        "chip_id":  chip_id,
        "s1_date":  event["s1_date"],
        "s2_date":  event["s2_date"],
        "rel_orbit_num": event["rel_orbit_num"],
        "orbit": event["orbit"],
        "min_lon":   round(b[0], 4),
        "min_lat":   round(b[1], 4),
        "max_lon":   round(b[2], 4),
        "max_lat":   round(b[3], 4),
        # "geometry":  geom
    })

# ── Export ────────────────────────────────────────────────────────────────────
out_df = pd.DataFrame(records)
result = pd.merge(filtered, out_df, on='chip_id', how='left')

# result.to_csv("usa_chips_with_metadata_15_percent_flood_cropland.csv", index=False)
print(f"\nSaved {len(result)} chips → usa_chips_with_metadata_15_percent_flood_cropland.csv")
print(result.to_string(index=False))

Processing 23 chips from filtered usa_flooded_cropland_chips_15%.csv CSV

Saved 23 chips → usa_chips_with_metadata_15_percent_flood_cropland.csv
    chip_id  flooded_crop_frac  flooded_crop_px    s1_date    s2_date  rel_orbit_num     orbit  min_lon  min_lat  max_lon  max_lat
 USA_430764              0.861            13289 2019/05/22 2019/05/22          136.0 ASCENDING -95.0231  39.6006 -94.9771  39.6466
 USA_955053              0.767            27479 2019/05/22 2019/05/22          136.0 ASCENDING -95.5290  38.5887 -95.4830  38.6347
  USA_58086              0.763            12130 2019/05/22 2019/05/22          136.0 ASCENDING -95.2070  38.5428 -95.1611  38.5887
  USA_86502              0.748            49655 2019/05/22 2019/05/22          136.0 ASCENDING -94.8851  39.2327 -94.8391  39.2787
 USA_170264              0.742            32290 2019/05/22 2019/05/22          136.0 ASCENDING -95.2990  38.5887 -95.2530  38.6347
 USA_595451              0.732            15028 2019/05/22 2019/05/22